# Air Quality Data Processing Pipeline
## From Raw Sensor Data to Features for Machine Learning

This notebook implements a complete **ETL + Machine Learning** pipeline for air quality monitoring data,
obtained from the Ukrainian Open Data Portal. The architecture is built around the principle of **dependency injection**:
each function receives its dependencies (DataFrame, sessions, configuration dictionaries) explicitly as arguments —
making each stage independently testable and reusable without hidden global state.

### Pipeline Overview
```
Web Scraping → Data Cleansing → DB Schema → ETL Loading → EDA → Feature Building → Model Training
```

---
> **Architectural Note:** The pipeline uses the *Snowflake* data warehouse schema internally,
> and then transforms the data into a wide, resampled format for machine learning.

## 1. Imports and Dependencies

All libraries are imported at the beginning. Grouping by purpose allows you to detect missing dependencies before running any subsequent cells.

In [ ]:
import os
import re
from pathlib import Path
from datetime import datetime

# ── Numerical & tabular ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# ── Database (SQLAlchemy ORM) ─────────────────────────────────────────────────
from sqlalchemy import (
    Column, Integer, BigInteger, Float,
    String, Boolean, DateTime, ForeignKey
)
from sqlalchemy import create_engine, URL
from sqlalchemy.orm import declarative_base, relationship, Session, sessionmaker

# ── Machine Learning ──────────────────────────────────────────────────────────
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

# ── Utilities ─────────────────────────────────────────────────────────────────
from typing import Dict, Any, List, Tuple
from IPython.display import clear_output, display

# ── Web scraping ──────────────────────────────────────────────────────────────
from bs4 import BeautifulSoup
import requests

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid")


## 2. Central configuration

Instead of scattering magic numbers and paths all over the laptop, **all parameters are concentrated in a single dictionary**.
Functions get this config through dependency injection, so changing the value here automatically propagates everywhere.

In [ ]:
CONFIG: Dict[str, Any] = {
    # ── Data sources ──────────────────────────────────────────────────────────
    "source_url"    : "https://opendata.gov.ua/dataset/air_monitor",
    "data_dir"      : "./data",
    "date_regex"    : r"\d{4}-\d{2}-\d{2}",

    # ── Database connection ────────────────────────────────────────────────────
    "db_driver"     : "mysql",
    "db_host"       : "db",       # replace with 127.0.0.1 for local MySQL
    "db_port"       : 3306,

    # ── ETL ────────────────────────────────────────────────────────────────────
    "batch_size"    : 5000,

    # ── ML pipeline ────────────────────────────────────────────────────────────
    "resample_freq" : "h",         # hourly resampling
    "forecast_horizon_h": 3,       # predict N hours ahead
    "pm25_clip_upper"   : 300,     # µg/m³ — WHO extreme threshold
    "pm10_clip_upper"   : 500,
    "interp_limit"      : 3,       # max hours to linearly interpolate
    "min_station_hours" : 1500,    # stations with fewer hours are dropped
    "min_continuity"    : 0.5,     # fraction of non-NaN hours required
    "top_k_neighbors"   : 3,       # spatial neighbours for gap-filling
    "test_size"         : 0.2,
    "cv_splits"         : 5,
    "use_pm10_feature"  : False,   # toggle PM10 as a predictor
}


## 3. Data collection (web scraping)

The output CSV files are published on the Ukrainian Open Data Portal. This section parses the portal's HTML page,
extracts the upload date and URL of each file, and then writes the files to `./data/`.

> **Dependency implementation:** `scrape_air_monitor_data` takes the URL and target directory as arguments,
> which allows for easy redirection to a mirror or test server without editing the function body.

In [ ]:
def scrape_air_monitor_data(source_url: str, data_dir: str, date_regex: str) -> None:
    """
    Downloads all CSV files listed on the open air monitoring data page.

    Parameters
    ----------
    source_url : str -- URL of the dataset index page.
    data_dir : str -- local directory to save the files.
    date_regex : str -- regular expression to extract the date string from the filename.
    """
    Path(data_dir).mkdir(parents=True, exist_ok=True)

    response = requests.get(source_url)
    soup = BeautifulSoup(response.text, "html.parser")

    for item in soup.find_all("div", {"class": "resource-item"}):
        # ── Search for file name for display ───────────────────────────────
        name_block = item.find("div", {"class": "data-resource-name-content"})
        link_tag   = name_block.find("a") if name_block else None
        if not link_tag:
            continue

        # ── Extracting a date substring for a file name ──────────────────────
        raw_title = link_tag.string or ""
        matches   = re.findall(date_regex, raw_title)
        if not matches:
            continue
        filename = Path(data_dir) / f"{matches[0]}.csv"

        # ── Search for download link ─────────────────────────────────────────
        download_tag = item.find("a", {"class": "data-resource-download"}, href=True)
        if not download_tag:
            continue
        table_link = download_tag.get("href")

        # ── Loading and saving ───────────────────────────────────────────────
        file_response = requests.get(table_link)
        print(f"[{file_response.status_code}] {filename.name}  ← {table_link}")

        if file_response.status_code == 200:
            filename.write_bytes(file_response.content)


# Start scraper (comment if files are already downloaded)
# scrape_air_monitor_data(
#     source_url=CONFIG["source_url"],
#     data_dir=CONFIG["data_dir"],
#     date_regex=CONFIG["date_regex"],
# )


## 4. Loading and Merging Data

### 4.1 Checking Column Compatibility

Before merging all CSV files, it is checked that each file has **the same column schema**.
If the new export adds or renames a column, the check will detect this immediately, preventing a silent incorrect merge.

In [ ]:
def check_if_columns_same(data_dir: str) -> bool:
    """
    Returns True if each CSV in data_dir has an identical set of column names.

    Parameters
    ----------
    data_dir : str -- directory with source CSV files.
    """
    seen_schemas = set()

    for filepath in Path(data_dir).rglob("*.csv"):
        try:
            sample = pd.read_csv(filepath, nrows=0)          # header only
        except UnicodeDecodeError:
            sample = pd.read_csv(filepath, nrows=0, encoding="cp1251")

        seen_schemas.add(tuple(sample.columns))

    return len(seen_schemas) == 1


columns_match = check_if_columns_same(CONFIG["data_dir"])
print("Всі файли мають однакові стовпці:", columns_match)


### 4.2 Concatenating all CSV files

Since all files have the same schema, they can be safely concatenated using `pd.concat`.

In [ ]:
def combine_tables(data_dir: str) -> pd.DataFrame:
    """
    Reads all CSV files from data_dir and concatenates them into a single DataFrame.

    Parameters
    ----------
    data_dir : str -- directory with source CSV files.

    Returns
    -------
    pd.DataFrame -- concatenated dataset with index reset.
    """
    files = list(Path(data_dir).rglob("*.csv"))
    frames = []
    for f in files:
        try:
            frames.append(pd.read_csv(f))
        except UnicodeDecodeError:
            frames.append(pd.read_csv(f, encoding="cp1251"))

    return pd.concat(frames, ignore_index=True)


df = combine_tables(CONFIG["data_dir"])
print(f"Розміри об'єднаного DataFrame: {df.shape}")
df.head()


## 5. 🧹 Data Cleanup

The output data contains a number of quality issues that need to be addressed before use:

| Issue | Fix |
|---|---|
| Missing measurement values ​​/ identifiers | Delete relevant rows |
| Stations with multiple conflicting names | Rename to canonical names |
| Stations with conflicting GPS coordinates | Replace with station-wide coordinates modality |
| Non-standard parameter codes (`SDS_P1`, `PM25`, …) | Reduce to canonical names (`PM10`, `PM2.5`, …) |
| Non-standard unit strings | Normalize to Unicode characters (`µg/m³`, `°C`, …) |
| Mixed date-time formats (ISO 8601 and legacy) | Parsing with `format='mixed'` |

### 5.1 Deleting rows with missing critical fields

In [ ]:
def prune_invalid_measurements(df: pd.DataFrame) -> pd.DataFrame:
    """
    Removes rows where the measurement value or parameter ID is missing.

    Parameters
    ----------
    df : pd.DataFrame -- the original merged dataset.

    Returns
    -------
    pd.DataFrame -- a cleaned copy with invalid rows removed.
    """
    df = df.copy()
    initial = len(df)

    df = df.dropna(subset=["stations_params_value", "stations_params_id"])

    dropped = initial - len(df)
    print(f"Видалено рядків  : {dropped:,}  ({dropped/initial:.1%})")
    print(f"Залишилось рядків: {len(df):,}")

    # Replace NaN with None for correct SQLAlchemy processing
    return df.replace({float("nan"): None})


df = prune_invalid_measurements(df)


### 5.2 Вирішення конфліктів назв станцій

Один `stations_id` повинен мати рівно одну `stations_name`. Функція нижче виявляє будь-які порушення;
якщо вони знайдені, виконується перейменування до стабільної канонічної назви.


In [ ]:
def identify_duplicate_names(df: pd.DataFrame) -> None:
    """
    Prints the station IDs associated with more than one name.

    Parameters
    ----------
    df : pd.DataFrame -- a dataset with columns 'stations_id' and 'stations_name'.
    """
    name_groups = df.groupby("stations_id")["stations_name"].unique()
    problematic = name_groups[name_groups.apply(len) > 1]

    if problematic.empty:
        print("Name check passed: no conflicts were found.")
        return

    print(f"WARN: {len(problematic)} station identifier(s) with conflicting names:\n")
    for sid, names in problematic.items():
        print(f"  ID {sid}: {', '.join(map(str, names))}")
        print("  " + "-" * 40)


identify_duplicate_names(df)


In [ ]:
def remap_conflicting_names(df: pd.DataFrame, stations_mapping: Dict[int, str]) -> pd.DataFrame:
    """
    Substitutes station names for the identifiers present in stations_mapping.

    Parameters
    ----------
    df : pd.DataFrame -- dataset to update.
    stations_mapping : Dict[int, str] -- mapping {station_id: canonical_name}.

    Returns
    -------
    pd.DataFrame -- copy with corrected station names.
    """
    df = df.copy()
    mask = df["stations_id"].isin(stations_mapping.keys())
    df.loc[mask, "stations_name"] = df.loc[mask, "stations_id"].map(stations_mapping)
    return df


# Display canonical names for stations with conflicting labels
STATION_NAME_MAP: Dict[int, str] = {
    256: "vinnytsia-256",
    281: "vinnytsia-281",
    315: "vinnytsia-315",
    90: "vinnytsia-90",
    271: "vinnytsia-271",
    767: "Соборна 36",
    1183: "Вишенька",
    246: "vinnytsia-246",
    274: "vinnytsia-274",
}

df = remap_conflicting_names(df, STATION_NAME_MAP)
identify_duplicate_names(df)


### 5.3 Стандартизація GPS-координат

Деякі станції мають незначно різні координати в різних експортах (похибки округлення, помилки введення).
Замінюємо всі значення координат для кожної станції на **моду** (найбільш часто спостережуване значення) -- статистично стійкий показник для такого типу шуму.


In [ ]:
def check_coordinate_consistency(df: pd.DataFrame) -> None:
    """
    Outputs stations whose Lat or Long columns contain more than one unique value.

    Parameters
    ----------
    df : pd.DataFrame -- dataset with columns 'stations_id', 'Lat', 'Long'.
    """
    stats = df.groupby("stations_id")[["Lat", "Long"]].nunique()
    bad   = stats[(stats["Lat"] > 1) | (stats["Long"] > 1)].index

    if bad.empty:
        print("Coordinate check passed: all stations have consistent coordinates.")
        return

    print(f"  {len(bad)} station(s) with inconsistent coordinates:\n")
    for sid in bad:
        lats  = df.loc[df["stations_id"] == sid, "Lat"].unique()
        longs = df.loc[df["stations_id"] == sid, "Long"].unique()
        print(f"  Station {sid} | Lat: {lats}  Long: {longs}")

In [ ]:
def standardize_coordinates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans, converts, and normalizes Lat/Long coordinates using the mode.

    Parameters
    ----------
    df : pd.DataFrame -- dataset to update.

    Returns
    -------
    pd.DataFrame -- the same DataFrame with cleaned coordinate columns.
    """
    df = df.copy()

    def _get_mode(series: pd.Series):
        m = series.mode()
        return m.iloc[0] if not m.empty else None

    for col in ["Lat", "Long"]:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.extract(r"(\d+\.\d+)")[0]
        df[col] = pd.to_numeric(df[col], errors="coerce").round(5)

    df[["Lat", "Long"]] = df.groupby("stations_id")[["Lat", "Long"]].transform(_get_mode)
    print("Coordinates cleaned and standardized.")
    return df


check_coordinate_consistency(df)
df = standardize_coordinates(df)
check_coordinate_consistency(df)   # check -- should output success message

### 5.4 Standardization of parameter codes and units

Sensors from different manufacturers report the same pollutant under different codes
(`SDS_P2`, `PM25`, `PM2.5`). All are brought to a single canonical dictionary.
Similarly, units come in different encodings (`ug/m3`, `μg/m³`) and are normalized
to the correct Unicode characters (`µg/m³`).

In [ ]:
# -- Canonical parameter code -> display name ---------------------------------
CANONICAL_PARAMS: Dict[str, str] = {
    # Particulate matter
    "SDS_P1": "PM10",   "SDS_P2": "PM2.5",
    "PMS_P0": "PM1.0",  "PMS_P1": "PM10",   "PMS_P2": "PM2.5",
    "PM0"   : "PM1.0",  "PM1"   : "PM1.0",  "PM25"  : "PM2.5",
    "PM100" : "PM10",   "PM1.0" : "PM1.0",  "PM2.5" : "PM2.5",
    "PM10"  : "PM10",
    # Gases
    "CO2": "CO2", "CO": "CO", "NO2": "NO2", "NO₂": "NO2",
    "O3" : "O3",  "O₃": "O3", "NH3": "NH3",
    "CH2O": "HCHO", "H2CO": "HCHO", "VOC": "VOC",
    # Meteorological parameters
    "TEMPERATURE": "Temperature",
    "HUMIDITY"   : "Humidity",
    "PRESSURE"   : "Pressure",
    # Radiation
    "RAD": "Radiation",
    # Aliases for specific sensors
    "A4": "CO", "E1": "NO2", "E3": "O3",
}

# -- Non-standard unit string -> Unicode canonical form ----------------------
UNIT_MAP: Dict[str, str] = {
    "ug/m3"   : "µg/m³", "мкг/м³": "µg/m³", "ug/m³": "µg/m³",
    "ppm"     : "ppm",   "ppb"    : "ppb",
    "%"       : "%",     "Rh"     : "%",
    "°C"      : "°C",   "C"      : "°C",
    "Pa"      : "Pa",   "hPa"    : "hPa",
    "mg/m3"   : "mg/m³",
    "uSv/h"   : "µSv/h",
}

In [ ]:
def resolve_canonical_parameter(raw_key: Any) -> str:
    """
    Converts a raw sensor parameter code into a canonical name.

    Parameters
    ----------
    raw_key : Any -- raw value from the 'stations_params_key' column.

    Returns
    -------
    str -- canonical name or uppercase string if no mapping exists.
    """
    if pd.isna(raw_key) or str(raw_key).lower() == "nan":
        return "UNKNOWN"

    key = str(raw_key).strip()

    # Remove everything before '=' sign (manufacturer prefix)
    if "=" in key:
        key = key.split("=")[-1]

    # Remove content in parentheses (units embedded in the code)
    key = re.sub(r"\(.*?\)", "", key)

    # Normalize separators and case
    key = key.replace(" ", "").replace("_", "").replace(".", "").upper()

    return CANONICAL_PARAMS.get(key, key)

In [ ]:
def unify_measurement_units(df: pd.DataFrame, unit_map: Dict[str, str]) -> pd.DataFrame:
    """
    Normalizes the 'stations_params_unit' column to Unicode canonical symbols.

    Parameters
    ----------
    df       : pd.DataFrame -- dataset to process.
    unit_map : Dict[str, str] -- mapping from raw unit strings to canonical ones.

    Returns
    -------
    pd.DataFrame -- copy with normalized unit column.
    """
    def _clean(u):
        if pd.isna(u):
            return "unknown"
        return unit_map.get(str(u).strip().replace("(", "").replace(")", ""), str(u).strip())

    df = df.copy()
    df["stations_params_unit"] = df["stations_params_unit"].apply(_clean)
    return df

In [ ]:
def standardize_dimension_attributes(
    df        : pd.DataFrame,
    canonical : Dict[str, str],
    unit_map  : Dict[str, str],
) -> pd.DataFrame:
    """
    Orchestrates the normalization of parameter codes and measurement units.

    Parameters
    ----------
    df        : pd.DataFrame -- dataset to clean.
    canonical : Dict[str, str] -- CANONICAL_PARAMS dictionary.
    unit_map  : Dict[str, str] -- UNIT_MAP dictionary.

    Returns
    -------
    pd.DataFrame -- fully normalized copy.
    """
    df = df.copy()
    df["stations_params_key"] = df["stations_params_key"].apply(resolve_canonical_parameter)
    df = unify_measurement_units(df, unit_map)
    # Fill missing names with canonical code
    df["stations_params_name"] = df["stations_params_name"].fillna(df["stations_params_key"])
    return df


df = standardize_dimension_attributes(df, CANONICAL_PARAMS, UNIT_MAP)
print("Unique parameter codes after normalization:")
print(sorted(df["stations_params_key"].unique()))

### 5.5 Parsing and unifying timestamps

The two timestamp columns (`stations_time` and `stations_params_time`) contain a mixture of ISO 8601 and legacy formats.
`format='mixed'` Pandas handles both transparently and converts everything to UTC.

In [ ]:
def prepare_dataframe_dates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Parses both date-time columns to UTC-aware pd.Timestamp labels.

    Parameters
    ----------
    df : pd.DataFrame -- original or partially cleaned DataFrame.

    Returns
    -------
    pd.DataFrame -- copy with both date columns converted to UTC.
    """
    df = df.copy()
    for col in ["stations_time", "stations_params_time"]:
        df[col] = pd.to_datetime(df[col], format="mixed", utc=True)
    return df


df = prepare_dataframe_dates(df)
print("Date column types after parsing:")
print(df[["stations_time", "stations_params_time"]].dtypes)

## 6. Database Schema (Snowflake Model)

The cleaned data is stored in the **Snowflake data store** schema:

```
DimUnit <-- DimParameter <--+
                            FactMeasurement
              DimStation <--+
```

- **`DimUnit`** -- canonical units
- **`DimParameter`** -- canonical pollution/weather parameters (with SCD Type 2 support)
- **`DimStation`** -- monitoring stations with GPS coordinates (with SCD Type 2 support)
- **`FactMeasurement`** -- one row per sensor measurement

SCD Type 2 columns (`valid_from`, `valid_to`, `is_current`) allow you to track the history of station renames or sensor reconfigurations without losing previous measurements.

In [ ]:
class DimUnit(Base):
    """Dimension table: measurement units (µg/m³, ppm, °C, …)."""
    __tablename__ = "dim_units"

    unit_key       = Column(Integer, primary_key=True, autoincrement=True)
    unit_name      = Column(String(64), nullable=False)
    unit_symbol    = Column(String(8))
    local_unit_name = Column(String(64))

    # Reverse reference: one unit -- many parameters
    parameters = relationship("DimParameter", back_populates="unit")

In [ ]:
class DimParameter(Base):
    """Dimension table: measured parameters (PM2.5, NO2, Temperature, …)."""
    __tablename__ = "dim_parameters"

    parameter_key  = Column(Integer, primary_key=True, autoincrement=True)
    parameter_code = Column(String(64), nullable=False)
    parameter_name = Column(String(64))
    local_name     = Column(String(64))
    unit_key       = Column(Integer, ForeignKey("dim_units.unit_key"))

    # SCD Type 2 -- tracking changes in parameter definitions over time
    valid_from = Column(DateTime, nullable=False)
    valid_to   = Column(DateTime)
    is_current = Column(Boolean, default=True)

    unit         = relationship("DimUnit", back_populates="parameters")
    measurements = relationship("FactMeasurement", back_populates="parameter")

In [ ]:
class DimStation(Base):
    """Dimension table: monitoring stations with GPS coordinates."""
    __tablename__ = "dim_stations"

    station_key     = Column(Integer, primary_key=True, autoincrement=True)
    station_id      = Column(Integer)
    station_name    = Column(String(128))
    latitude        = Column(Float)
    longitude       = Column(Float)
    timezone_offset = Column(Integer)

    # SCD Type 2 -- tracking station renames / relocations
    valid_from = Column(DateTime, nullable=False)
    valid_to   = Column(DateTime)
    is_current = Column(Boolean, default=True)

    measurements = relationship("FactMeasurement", back_populates="station")

In [ ]:
class FactMeasurement(Base):
    """
    Fact table: one row per sensor measurement.

    Foreign keys reference dimension tables for station, parameter, and unit.
    Time dimension is embedded directly (without a separate DimTime table).
    """
    __tablename__ = "fact_measurements"

    measurement_id = Column(BigInteger, primary_key=True, autoincrement=True)

    # ── Foreign keys ──────────────────────────────────────────────────────────
    station_key   = Column(Integer, ForeignKey("dim_stations.station_key"),   nullable=False)
    parameter_key = Column(Integer, ForeignKey("dim_parameters.parameter_key"), nullable=False)

    # ── Measured values ───────────────────────────────────────────────────────
    value           = Column(Float)
    quality_ratio   = Column(Float, nullable=True)   # calibration confidence
    pollution_level = Column(Integer)                # optional AQI band

    # ── Time dimension ────────────────────────────────────────────────────────
    measurement_timestamp = Column(DateTime, nullable=False, index=True)
    offset_minutes        = Column(Integer)           # UTC offset of station

    station   = relationship("DimStation",   back_populates="measurements")
    parameter = relationship("DimParameter", back_populates="measurements")

### 6.1 Creating a physical database

Adjust the `host` value in `CONFIG` to suit your deployment environment:

| Script | `db_host` value |
|---|---|
| Local MySQL | `127.0.0.1` |
| Docker Compose | service name (e.g. `db`) |
| Remote server | hostname or IP |

Credentials are read from environment variables -- never hardcoded.

In [ ]:
def create_db_engine(config: Dict[str, Any]):
    """
    Creates a SQLAlchemy engine from the central CONFIG dictionary.

    Parameters
    ----------
    config : Dict[str, Any] -- must contain keys: db_driver, db_host, db_port.
                              Credentials: MYSQL_USER, MYSQL_PASSWORD, MYSQL_DATABASE.

    Returns
    -------
    sqlalchemy.engine.Engine
    """
    url = URL.create(
        drivername=config["db_driver"],
        username=os.getenv("MYSQL_USER"),
        password=os.getenv("MYSQL_PASSWORD"),
        host=config["db_host"],
        port=config["db_port"],
        database=os.getenv("MYSQL_DATABASE"),
    )
    return create_engine(url)


engine = create_db_engine(CONFIG)
Base.metadata.create_all(engine)   # Creates tables if they don't already exist
print("Database schema created (or already exists).")

## 7. 🔄 ETL pipeline

Each `transform_and_load_*` function follows the same pattern:

1. **Extract** unique dimension records from the DataFrame
2. **Check** for a matching record in the database (idempotent upsert)
3. **Insert** new records and `flush()` to get surrogate keys
4. **Return** the `{business-key -> surrogate_key}` mapping for the next step

All functions receive an argument `session: Session` -- the point of dependency injection.

In [ ]:
def transform_and_load_units(df: pd.DataFrame, session: Session) -> Dict[str, int]:
    """
    Performs upsert of unique measurement units into DimUnit.

    Parameters
    ----------
    df      : pd.DataFrame -- cleaned dataset.
    session : Session      -- active SQLAlchemy session (injected from outside).

    Returns
    -------
    Dict[str, int] -- mapping {unit_symbol -> unit_key}.
    """
    unique_units = df[["stations_params_unit", "stations_params_localUnit"]].drop_duplicates()
    unit_map: Dict[str, int] = {}

    for _, row in unique_units.iterrows():
        symbol = row["stations_params_unit"]
        unit   = session.query(DimUnit).filter_by(unit_symbol=symbol).first()

        if not unit:
            unit = DimUnit(
                unit_name      = symbol,
                unit_symbol    = symbol,
                local_unit_name= row["stations_params_localUnit"],
            )
            session.add(unit)
            session.flush()   # populate unit.unit_key without committing

        unit_map[unit.unit_symbol] = unit.unit_key

    return unit_map

In [ ]:
def transform_and_load_parameters(
    df      : pd.DataFrame,
    session : Session,
    unit_map: Dict[str, int],
) -> Dict[str, int]:
    """
    Performs upsert of unique measurement parameters into DimParameter.

    Parameters
    ----------
    df       : pd.DataFrame   -- cleaned dataset.
    session  : Session        -- active SQLAlchemy session (injected from outside).
    unit_map : Dict[str, int] -- mapping {unit_symbol -> unit_key}.

    Returns
    -------
    Dict[str, int] -- mapping {parameter_code -> parameter_key}.
    """
    cols = ["stations_params_key", "stations_params_name",
            "stations_params_localName", "stations_params_unit"]
    unique_params = df[cols].drop_duplicates()
    param_map: Dict[str, int] = {}

    for _, row in unique_params.iterrows():
        code_val = row["stations_params_key"]
        param    = session.query(DimParameter).filter_by(parameter_code=code_val).first()

        if not param:
            param = DimParameter(
                parameter_code = code_val,
                parameter_name = row["stations_params_name"],
                local_name     = row["stations_params_localName"],
                unit_key       = unit_map.get(row["stations_params_unit"]),
                valid_from     = datetime.now(),
                is_current     = True,
            )
            session.add(param)
            session.flush()

        param_map[param.parameter_code] = param.parameter_key

    return param_map

In [ ]:
def transform_and_load_stations(df: pd.DataFrame, session: Session) -> Dict[int, int]:
    """
    Performs upsert of unique stations into DimStation.

    Parameters
    ----------
    df      : pd.DataFrame -- cleaned dataset.
    session : Session      -- active SQLAlchemy session (injected from outside).

    Returns
    -------
    Dict[int, int] -- mapping {business_station_id -> surrogate_station_key}.
    """
    cols = ["stations_id", "stations_name", "Lat", "Long", "stations_offset"]
    unique_stations = df[cols].drop_duplicates()
    station_map: Dict[int, int] = {}

    for _, row in unique_stations.iterrows():
        station = session.query(DimStation).filter_by(
            station_id=row["stations_id"], is_current=True
        ).first()

        if not station:
            station = DimStation(
                station_id     = row["stations_id"],
                station_name   = row["stations_name"],
                latitude       = row["Lat"],
                longitude      = row["Long"],
                timezone_offset= row["stations_offset"],
                valid_from     = datetime.now(),
                is_current     = True,
            )
            session.add(station)
            session.flush()

        station_map[station.station_id] = station.station_key

    return station_map

In [ ]:
def load_fact_measurements(
    df         : pd.DataFrame,
    session    : Session,
    station_map: Dict[int, int],
    param_map  : Dict[str, int],
    batch_size : int = 5000,
) -> None:
    """
    Bulk inserts measurement records into FactMeasurement in batches.

    Parameters
    ----------
    df          : pd.DataFrame   -- fully processed dataset.
    session     : Session        -- active SQLAlchemy session (injected from outside).
    station_map : Dict[int, int] -- {business_id -> surrogate_key} for stations.
    param_map   : Dict[str, int] -- {parameter_code -> surrogate_key}.
    batch_size  : int            -- rows per commit cycle (default 5,000).
    """
    buffer: List[FactMeasurement] = []
    n_added  = 0
    n_total  = len(df)

    for _, row in df.iterrows():
        ts = row["stations_params_time"]
        if pd.isna(ts):
            continue

        s_key = station_map.get(row["stations_id"])
        p_key = param_map.get(row["stations_params_key"])
        if s_key is None or p_key is None:
            continue   # orphaned record -- skip

        buffer.append(FactMeasurement(
            station_key           = s_key,
            parameter_key         = p_key,
            value                 = row["stations_params_value"],
            quality_ratio         = row["stations_params_cr"],
            pollution_level       = row["stations_params_level"],
            measurement_timestamp = ts,
            offset_minutes        = row["stations_params_offset"],
        ))

        if len(buffer) >= batch_size:
            session.bulk_save_objects(buffer)
            session.commit()
            buffer.clear()
            session.expunge_all()   # free ORM identity map memory
            n_added += batch_size
            print(f"\rInserted: {n_added:,} / {n_total:,}", end="")

    # Flush any remaining records
    if buffer:
        session.bulk_save_objects(buffer)
        session.commit()
        session.expunge_all()
        buffer.clear()

    print(f"\nFact table loading completed.")

### 7.1 Orchestrator -- `run_pipeline`

The orchestrator connects the four load functions in the correct order of dependencies and wraps
everything in a single transaction: if any step throws an exception, the session is rolled back.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional


# -- Step-function type aliases ------------------------------------------------
LoadUnitsFn      = Callable[[pd.DataFrame, Session], Dict[str, int]]
LoadParamsFn     = Callable[[pd.DataFrame, Session, Dict[str, int]], Dict[str, int]]
LoadStationsFn   = Callable[[pd.DataFrame, Session], Dict[int, int]]
LoadFactsFn      = Callable[[pd.DataFrame, Session, Dict[int, int], Dict[str, int], int], None]


@dataclass
class ETLPipelineSteps:
    """
    Container for all replaceable ETL step functions.

    Each field accepts any callable with the corresponding signature, allowing
    easy replacement of individual steps without changing the orchestrator.
    """
    load_units    : LoadUnitsFn    = field(default=transform_and_load_units)
    load_params   : LoadParamsFn   = field(default=transform_and_load_parameters)
    load_stations : LoadStationsFn = field(default=transform_and_load_stations)
    load_facts    : LoadFactsFn    = field(default=load_fact_measurements)

In [ ]:
def run_pipeline(
    df         : pd.DataFrame,
    session    : Session,
    batch_size : int              = 5000,
    steps      : Optional[ETLPipelineSteps] = None,
) -> None:
    """
    Orchestrates the complete ETL load in dependency order with full dependency injection.

    Load order:
      1. Measurement units -- no dependencies
      2. Parameters        -- depend on units
      3. Stations          -- no dependencies
      4. Fact records      -- depend on parameters and stations

    Parameters
    ----------
    df         : pd.DataFrame     -- fully cleaned DataFrame.
    session    : Session          -- SQLAlchemy session (injected from outside).
    batch_size : int              -- rows per commit cycle.
    steps      : ETLPipelineSteps -- step functions (defaults to production implementations).
    """
    if steps is None:
        steps = ETLPipelineSteps()

    try:
        print("1. Loading measurement units ...")
        u_map = steps.load_units(df, session)

        print("2. Loading measurement parameters ...")
        p_map = steps.load_params(df, session, u_map)

        print("3. Loading stations ...")
        s_map = steps.load_stations(df, session)

        print("4. Loading measurements (fact table) ...")
        steps.load_facts(df, session, s_map, p_map, batch_size)

        print("\nETL pipeline completed successfully.")

    except Exception as exc:
        session.rollback()
        print(f"\nPipeline failed with error: {exc}")
        raise

In [ ]:
# -- Execution ----------------------------------------------------------------
SessionFactory = sessionmaker(engine)
session = SessionFactory()

# run_pipeline(df, session, batch_size=CONFIG["batch_size"])

## 8. Exploratory Data Analysis (EDA)

Before building ML models, we examine the data: distribution, temporal patterns, correlations, and spatial distribution.

In [ ]:
# Ensure time column is sorted for all plots
df = df.sort_values("stations_time")

print(df.info())
print("\nMissing values by column:")
print(df.isnull().sum())

### 8.1 Number of measurements by station

Which stations report the most data? Stations with very few records are unreliable for ML.

In [ ]:
plt.figure(figsize=(10, 8))
df["stations_name"].value_counts().plot(kind="barh")
plt.title("Number of recorded measurements by station")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

### 8.2 Distribution of Pollutant Values

Box plot (log scale) shows the spread and emissions for each pollutant.

In [ ]:
POLLUTANTS = ["PM2.5", "PM10", "CO2", "Temperature", "Humidity"]

plt.figure(figsize=(15, 8))
sns.boxplot(
    data=df[df["stations_params_key"].isin(POLLUTANTS)],
    x="stations_params_key",
    y="stations_params_value",
)
plt.yscale("log")
plt.title("Distribution of main pollutants (logarithmic scale)")
plt.xlabel("Parameter")
plt.ylabel("Value (log scale)")
plt.tight_layout()
plt.show()

### 8.3 PM2.5 Time Series -- Top 5 Stations

Daily averages make the graph readable while preserving the trend signal.

In [ ]:
top_stations = df["stations_name"].value_counts().nlargest(5).index
pm25_data   = df[
    (df["stations_params_key"] == "PM2.5") &
    (df["stations_name"].isin(top_stations))
]

pm25_daily = (
    pm25_data
    .groupby(["stations_name", pd.Grouper(key="stations_time", freq="D")])["stations_params_value"]
    .mean()
    .reset_index()
)

fig = px.line(
    pm25_daily,
    x="stations_time", y="stations_params_value", color="stations_name",
    title="Daily average PM2.5 values -- Top 5 stations",
    labels={"stations_params_value": "PM2.5 (µg/m³)", "stations_time": "Date"},
)
fig.show()

### 8.4 Heatmap of Correlations Between Pollutants

Uses a sample of 100,000 rows to manage memory usage.

In [ ]:
df_sample = df.sample(100_000, random_state=42)

pivot_corr = df_sample.pivot_table(
    index   = ["stations_id", "stations_time"],
    columns = "stations_params_key",
    values  = "stations_params_value",
    aggfunc = "mean",
)

plt.figure(figsize=(12, 10))
sns.heatmap(pivot_corr.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation heatmap -- Pollutants and meteorological factors")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Ensure the value column is numeric (convert strings to NaN, then drop or handle them)
df["stations_params_value"] = pd.to_numeric(df["stations_params_value"], errors='coerce')

# 2. Group and aggregate
map_data = (
    df[df["stations_params_key"] == "PM2.5"]
    .groupby("stations_name")
    .agg(
        Lat=("Lat", "first"), 
        Long=("Long", "first"),
        pm25_mean=("stations_params_value", "mean")
    )
    .reset_index()
)

# 3. Double-check: Drop rows where pm25_mean might be NaN to avoid plotting errors
map_data = map_data.dropna(subset=["pm25_mean"])

# 4. Use px.scatter_map (the updated version of scatter_mapbox)
fig = px.scatter_map(
    map_data,
    lat="Lat", 
    lon="Long",
    color="pm25_mean", 
    size="pm25_mean",
    hover_name="stations_name",
    color_continuous_scale=px.colors.sequential.Reds,
    size_max=15, 
    zoom=10,
    map_style="carto-positron", # Note: mapbox_style becomes map_style in the new function
    title="Average PM2.5 concentration by station location",
)

fig.show()

### 8.7 Добовий паттерн PM2.5 (за годинами доби)


In [ ]:
df["hour"] = df["stations_time"].dt.hour

hourly_pm = (
    df[df["stations_params_key"] == "PM2.5"]
    .groupby("hour")["stations_params_value"]
    .median()
)

plt.figure(figsize=(10, 5))
hourly_pm.plot(kind="line", marker="o", color="teal")
plt.title("Median PM2.5 level by hour of day (diurnal pattern)")
plt.xlabel("Hour (UTC)")
plt.ylabel("PM2.5 median (µg/m³)")
plt.xticks(range(0, 24))
plt.grid(True)
plt.tight_layout()
plt.show()

## 9. Machine Learning Pipeline

### Design Solutions

| Solutions | Rationale |
|---|---|
| **Goal:** PM2.5 in 3 hours | Short enough for practical use |
| **Resampling to hourly data** | Irregular intervals break time patterns |
| **Lags (1h -- 72h)** | Autocorrelation -- strongest predictor |
| **Cyclic time encoding** (sin/cos) | Preserves the circular nature of time of day |
| **Spatial neighbors** | Neighboring stations -- leading indicators |
| **RobustScaler + log1p goal** | Reduces the impact of extreme emissions |
| **TimeSeriesSplit CV** | Prevents data leakage from the future |

### 9.1 Station Ranking

Selecting stations with sufficient data density for reliable training.

In [ ]:
def rank_stations(df: pd.DataFrame, min_hours: int = 1000) -> pd.DataFrame:
    """
    Ranks stations by the number of valid hourly PM2.5 measurements.

    Parameters
    ----------
    df        : pd.DataFrame -- dataset in long format.
    min_hours : int          -- minimum number of valid measurements.

    Returns
    -------
    pd.DataFrame -- ranked table with columns [station, count].
    """
    stats = []
    for station in df["stations_name"].unique():
        pm = df[(df["stations_name"] == station) & (df["stations_params_key"] == "PM2.5")]
        if len(pm) >= min_hours:
            stats.append({"station": station, "count": len(pm)})

    return pd.DataFrame(stats).sort_values("count", ascending=False).reset_index(drop=True)


ranking      = rank_stations(df, min_hours=CONFIG["min_station_hours"])
best_station = ranking.iloc[0]["station"]
print(f"Top stations:\n{ranking.head(10)}\n")
print(f"-> Selected station for analysis: {best_station}")

### 9.2 Building a wide (pivoted) multi-station table

In [ ]:
def build_multi_station_table(df: pd.DataFrame, freq: str = "h") -> pd.DataFrame:
    """
    Pivots the long-format dataset into a wide table and resamples to a fixed frequency.

    Parameters
    ----------
    df   : pd.DataFrame -- cleaned dataset in long format.
    freq : str          -- resampling frequency (default 'h' = hourly).

    Returns
    -------
    pd.DataFrame -- wide table indexed by time labels.
    """
    df = df.copy()
    df["stations_params_value"] = pd.to_numeric(df["stations_params_value"], errors="coerce")

    pivot = df.pivot_table(
        index   = "stations_time",
        columns = ["stations_name", "stations_params_key"],
        values  = "stations_params_value",
        aggfunc = "mean",
    )
    return pivot.resample(freq).mean()


pivot = build_multi_station_table(df, freq=CONFIG["resample_freq"])
print("Wide table dimensions:", pivot.shape)

### 9.3 Reliable Station Filtering and Spatial Gap Filling

In [ ]:
def get_pm_matrix(pivot: pd.DataFrame) -> pd.DataFrame:
    """Extracts the PM2.5 slice from a multi-level pivot table."""
    return pivot.xs("PM2.5", level=1, axis=1)

In [ ]:
def filter_good_stations(
    pm_matrix      : pd.DataFrame,
    min_hours      : int   = 1500,
    min_continuity : float = 0.5,
) -> pd.DataFrame:
    """
    Returns a ranked DataFrame of stations meeting data quality thresholds.

    Parameters
    ----------
    pm_matrix      : pd.DataFrame -- wide PM2.5 matrix.
    min_hours      : int          -- minimum number of non-NaN hours.
    min_continuity : float        -- required fraction of non-NaN hours (0-1).
    """
    stats = []
    for station in pm_matrix.columns:
        valid      = pm_matrix[station].notna().sum()
        continuity = valid / len(pm_matrix)
        if valid >= min_hours and continuity >= min_continuity:
            stats.append({"station": station, "valid": valid, "continuity": continuity})

    return (
        pd.DataFrame(stats)
        .sort_values(["valid", "continuity"], ascending=False)
        .reset_index(drop=True)
    )


In [ ]:
def fill_with_neighbors(pm_matrix: pd.DataFrame, top_k: int = 3) -> pd.DataFrame:
    """
    Fills gaps in a PM2.5 series using weighted average of the most correlated neighbors.

    Parameters
    ----------
    pm_matrix : pd.DataFrame -- wide PM2.5 matrix.
    top_k     : int          -- number of nearest neighbors.

    Returns
    -------
    pd.DataFrame -- copy of the matrix with gaps filled.
    """
    filled = pm_matrix.copy()
    corr   = pm_matrix.corr()

    for station in pm_matrix.columns:
        neighbors = corr[station].drop(station).sort_values(ascending=False).head(top_k)
        mask      = filled[station].isna()

        for t in pm_matrix.index[mask]:
            vals, weights = [], []
            for nbr, w in neighbors.items():
                val = pm_matrix.loc[t, nbr]
                if not pd.isna(val):
                    vals.append(val)
                    weights.append(max(w, 0.0))   # clip negative correlations

            if vals and sum(weights) > 0:
                filled.loc[t, station] = np.average(vals, weights=weights)

    return filled

In [ ]:
pm_matrix = get_pm_matrix(pivot)
stats_df  = filter_good_stations(
    pm_matrix,
    min_hours      = CONFIG["min_station_hours"],
    min_continuity = CONFIG["min_continuity"],
)
pm_filled = fill_with_neighbors(pm_matrix[stats_df["station"]], top_k=CONFIG["top_k_neighbors"])

print("Quality stations:\n", stats_df.head(10))

### 9.4 Feature Engineering

The most important step for model performance. It creates:
- **Dense lag features** -- PM2.5 values ​​for t-1 h ... t-24 h, t-48 h, t-72 h
- **Moving statistics** -- mean and standard deviation for 3 h, 6 h, 12 h
- **Cyclic time features** -- sin/cos encoding of hour and day of week
- **Meteorological dynamics** -- delta and moving average for Temperature, Humidity, Pressure
- **Spatial context** -- shifted PM2.5 values ​​from correlated neighboring stations

In [ ]:
def select_stations(
    stats_df   : pd.DataFrame,
    pm_filled  : pd.DataFrame,
    top_k      : int = 3,
) -> Tuple[str, List[str]]:
    """
    Selects the main target station and the most correlated neighbors.

    Parameters
    ----------
    stats_df  : pd.DataFrame -- ranked station quality table.
    pm_filled : pd.DataFrame -- PM2.5 matrix with gaps filled.
    top_k     : int          -- number of neighbors to include.

    Returns
    -------
    Tuple[str, List[str]] -- (target_station_name, [neighbor_names]).
    """
    target   = stats_df.iloc[0]["station"]
    corr     = pm_filled.corr()[target].drop(target)
    related  = corr.sort_values(ascending=False).head(top_k).index.tolist()
    return target, related

In [ ]:
def build_features(
    pivot         : pd.DataFrame,
    pm_filled     : pd.DataFrame,
    target        : str,
    related       : List[str],
    forecast_h    : int  = 3,
    include_pm10  : bool = False,
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Builds feature matrix X and target vector y for ML models.

    Parameters
    ----------
    pivot        : pd.DataFrame -- full wide table.
    pm_filled    : pd.DataFrame -- PM2.5 matrix with gaps filled.
    target       : str          -- target station name.
    related      : List[str]    -- neighbor station names.
    forecast_h   : int          -- forecast horizon in hours.
    include_pm10 : bool         -- whether to include PM10 as a feature.

    Returns
    -------
    (X, y) : Tuple[pd.DataFrame, pd.Series]
    """
    y = pm_filled[target]

    # Start with all meteorological columns for the target station
    X = pivot[target].copy()
    if not include_pm10:
        X = X.drop(columns=["PM10"], errors="ignore")
    X = X.drop(columns=["PM2.5"], errors="ignore")   # prevent target leakage

    # -- Shift target forward in time (predicting the future) -----------------
    y_target      = y.shift(-forecast_h)
    y_target.name = "target"

    # -- Lag features ----------------------------------------------------------
    for lag in list(range(1, 25)) + [48, 72]:
        X[f"pm_lag_{lag}h"] = y.shift(lag)

    # -- Rolling statistics ----------------------------------------------------
    X["pm_roll_mean_3h"]  = y.shift(1).rolling(3).mean()
    X["pm_roll_mean_6h"]  = y.shift(1).rolling(6).mean()
    X["pm_roll_mean_12h"] = y.shift(1).rolling(12).mean()
    X["pm_roll_std_6h"]   = y.shift(1).rolling(6).std()

    # -- Cyclical time encoding ------------------------------------------------
    X["hour_sin"] = np.sin(2 * np.pi * X.index.hour / 24.0)
    X["hour_cos"] = np.cos(2 * np.pi * X.index.hour / 24.0)
    X["dow_sin"]  = np.sin(2 * np.pi * X.index.dayofweek / 7.0)
    X["dow_cos"]  = np.cos(2 * np.pi * X.index.dayofweek / 7.0)

    # -- Meteorological dynamics -----------------------------------------------
    for met_col in ["Temperature", "Humidity", "Pressure"]:
        if met_col in X.columns:
            X[f"{met_col}_delta_3h"]  = X[met_col].diff(3)
            X[f"{met_col}_delta_6h"]  = X[met_col].diff(6)
            X[f"{met_col}_roll_12h"]  = X[met_col].rolling(12).mean()

    if "Temperature" in X.columns and "Humidity" in X.columns:
        X["temp_x_humidity"] = X["Temperature"] * X["Humidity"]

    # -- Spatial context (neighbouring stations) -------------------------------
    for nbr in related:
        X[f"{nbr}_pm_lag_1h"] = pm_filled[nbr].shift(1)

    # -- Final clean-up --------------------------------------------------------
    combined = pd.concat([X, y_target], axis=1)
    combined = combined[combined["target"].notna()]
    combined = combined.ffill().bfill()

    return combined.drop(columns=["target"]), combined["target"]

In [ ]:
target_station, related_stations = select_stations(
    stats_df, pm_filled, top_k=CONFIG["top_k_neighbors"]
)

X, y = build_features(
    pivot, pm_filled, target_station, related_stations,
    forecast_h   = CONFIG["forecast_horizon_h"],
    include_pm10 = CONFIG["use_pm10_feature"],
)

print(f"Target station  : {target_station}")
print(f"Neighbor stations  : {related_stations}")
print(f"Feature matrix    : {X.shape}")

### 9.5 Splitting into training and test samples (chronological)

In [ ]:
def time_split(
    X         : pd.DataFrame,
    y         : pd.Series,
    test_size : float = 0.2,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
    Splits data chronologically to prevent future data leakage.

    Parameters
    ----------
    X, y      : feature matrix and target vector (with time index).
    test_size : float -- fraction of data for testing.

    Returns
    -------
    X_train, X_test, y_train, y_test
    """
    split = int(len(X) * (1 - test_size))
    return X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:]


X_train, X_test, y_train, y_test = time_split(X, y, test_size=CONFIG["test_size"])
tscv = TimeSeriesSplit(n_splits=CONFIG["cv_splits"])

print(f"Training set: {len(X_train):,} rows  |  Test set: {len(X_test):,} rows")

## 10. Model Training

Each model is wrapped in a `TransformedTargetRegressor` pipeline:

```
RobustScaler -> [Model] -> log1p(y) / expm1(y_hat)
```

- **RobustScaler** -- uses median/IQR, so outliers don't distort scaling
- **Log1p Transformation** -- compresses the right tail of the PM2.5 distribution
- **GridSearchCV with TimeSeriesSplit** -- tunes hyperparameters without future leakage

In [ ]:
def train_tuned_models(
    X_train : pd.DataFrame,
    y_train : pd.Series,
    tscv    : TimeSeriesSplit,
) -> Dict[str, Any]:
    """
    Trains and tunes Ridge, XGBoost, and RandomForest using GridSearchCV,
    then builds a VotingRegressor ensemble with the best versions.

    Parameters
    ----------
    X_train : pd.DataFrame    -- training features.
    y_train : pd.Series       -- target vector.
    tscv    : TimeSeriesSplit -- cross-validation splitter (injected from outside).

    Returns
    -------
    Dict[str, Any] -- {model_name: trained estimator}.
    """
    def _wrap(model_obj):
        """Wraps the model in a standard scaling + log-transformation pipeline."""
        return TransformedTargetRegressor(
            regressor     = Pipeline([("scaler", RobustScaler()), ("model", model_obj)]),
            func          = np.log1p,
            inverse_func  = np.expm1,
        )

    # -- 1. Ridge Regression ---------------------------------------------------
    grid_ridge = GridSearchCV(
        _wrap(Ridge()),
        {"regressor__model__alpha": [0.1, 1.0, 10.0, 100.0]},
        cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1,
    )

    # -- 2. XGBoost ------------------------------------------------------------
    grid_xgb = GridSearchCV(
        _wrap(xgb.XGBRegressor(objective="reg:squarederror", random_state=42, verbosity=0)),
        {
            "regressor__model__n_estimators": [100, 300],
            "regressor__model__max_depth"   : [3, 5],
            "regressor__model__learning_rate": [0.01, 0.1],
        },
        cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1,
    )

    # -- 3. Random Forest ------------------------------------------------------
    grid_rf = GridSearchCV(
        _wrap(RandomForestRegressor(random_state=42)),
        {
            "regressor__model__n_estimators"  : [100, 200],
            "regressor__model__max_depth"     : [10, 20, None],
            "regressor__model__min_samples_leaf": [1, 4],
        },
        cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1,
    )

    print("Tuning Ridge regression ...")
    grid_ridge.fit(X_train, y_train)

    print("Tuning XGBoost ...")
    grid_xgb.fit(X_train, y_train)

    print("Tuning Random Forest ...")
    grid_rf.fit(X_train, y_train)

    # -- Ensemble from best individual models ----------------------------------
    best_ridge = grid_ridge.best_estimator_.regressor_.named_steps["model"]
    best_xgb   = grid_xgb.best_estimator_.regressor_.named_steps["model"]
    best_rf    = grid_rf.best_estimator_.regressor_.named_steps["model"]

    ensemble_pipe = _wrap(VotingRegressor([
        ("ridge", best_ridge),
        ("xgb",   best_xgb),
        ("rf",    best_rf),
    ]))
    print("Training ensemble ...")
    ensemble_pipe.fit(X_train, y_train)

    print("\nAll models trained.")
    return {
        "Ridge_Tuned"     : grid_ridge.best_estimator_,
        "XGBoost_Tuned"   : grid_xgb.best_estimator_,
        "RandomForest_Tuned": grid_rf.best_estimator_,
        "Ensemble"        : ensemble_pipe,
    }


models = train_tuned_models(X_train, y_train, tscv)

### 9.6 ML Pipeline Orchestrator -- `run_ml_pipeline`

Dependency-injected orchestrator for a full ML pipeline.

> **Fix:** The `evaluate` field in `MLPipelineSteps` now defaults to `None`.
> The `evaluate` function is defined below (section 11); `field(default=evaluate)` was causing a `NameError`
> when loading this cell, which would cause execution to hang. The orchestrator now
> substitutes `evaluate` dynamically when called.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional, Tuple


# -- Type aliases for step functions ------------------------------------------
RankStationsFn    = Callable[[pd.DataFrame, int], pd.DataFrame]
BuildTableFn      = Callable[[pd.DataFrame, str], pd.DataFrame]
FilterStationsFn  = Callable[[pd.DataFrame, int, float], pd.DataFrame]
FillGapsFn        = Callable[[pd.DataFrame, int], pd.DataFrame]
SelectStationsFn  = Callable[[pd.DataFrame, pd.DataFrame, int], Tuple[str, list]]
BuildFeaturesFn   = Callable[..., Tuple[pd.DataFrame, pd.Series]]
SplitFn           = Callable[[pd.DataFrame, pd.Series, float], Tuple]
TrainFn           = Callable[[pd.DataFrame, pd.Series, "TimeSeriesSplit"], Dict[str, Any]]
EvaluateFn        = Callable[[Dict[str, Any], pd.DataFrame, pd.Series], pd.DataFrame]

#### Container for all replaceable ML pipeline step functions.

The ``evaluate`` field defaults to None because the ``evaluate`` function
is defined in section 11 (later in the notebook). Using ``field(default=evaluate)``
would raise a NameError when executing this cell, causing it to hang without
returning a result. The orchestrator ``run_ml_pipeline`` dynamically supplies
the correct value.

In [ ]:
@dataclass
class MLPipelineSteps:
    rank_stations    : RankStationsFn    = field(default=rank_stations)
    build_table      : BuildTableFn      = field(default=build_multi_station_table)
    filter_stations  : FilterStationsFn  = field(default=filter_good_stations)
    fill_gaps        : FillGapsFn        = field(default=fill_with_neighbors)
    select_stations  : SelectStationsFn  = field(default=select_stations)
    build_features   : BuildFeaturesFn   = field(default=build_features)
    split            : SplitFn           = field(default=time_split)
    train            : TrainFn           = field(default=train_tuned_models)
    # evaluate = None by default: function defined later in the notebook
    evaluate         : Optional[EvaluateFn] = field(default=None)


@dataclass
class MLPipelineResult:
    """Typed container returned by ``run_ml_pipeline``."""
    models          : Dict[str, Any]
    metrics         : pd.DataFrame
    X_train         : pd.DataFrame
    X_test          : pd.DataFrame
    y_train         : pd.Series
    y_test          : pd.Series
    target_station  : str
    related_stations: list

#### Orchestrates the complete ML pipeline with full dependency injection.

**Pipeline steps**
1. rank_stations   -- ranks stations by PM2.5 data density
2. build_table     -- pivots data into wide hourly table
3. filter_stations -- discards stations below quality thresholds
4. fill_gaps       -- spatially fills gaps via neighbors
5. select_stations -- selects target station and neighbors
6. build_features  -- builds feature matrix X and target vector y
7. split           -- chronological train/test split
8. train           -- trains and tunes all models
9. evaluate        -- computes R2, MAE, RMSE on test set

**Parameters**

- df     : pd.DataFrame      -- cleaned dataset in long format.
- config : Dict[str, Any]    -- central CONFIG dictionary.
- steps  : MLPipelineSteps   -- step functions (defaults to production).

**Returns**

MLPipelineResult -- container with models, metrics, and data.

In [ ]:
def run_ml_pipeline(
    df     : pd.DataFrame,
    config : Dict[str, Any],
    steps  : Optional[MLPipelineSteps] = None,
) -> MLPipelineResult:
    if steps is None:
        steps = MLPipelineSteps()

    # Dynamically supply evaluate (resolves definition order issue)
    eval_fn = steps.evaluate if steps.evaluate is not None else evaluate

    tscv = TimeSeriesSplit(n_splits=config["cv_splits"])

    print("1. Ranking stations ...")
    ranking = steps.rank_stations(df, min_hours=config["min_station_hours"])
    print(f"   -> {len(ranking)} stations meet the criteria")

    print("2. Building wide hourly table ...")
    pivot = steps.build_table(df, freq=config["resample_freq"])
    print(f"   -> dimensions: {pivot.shape}")

    print("3. Filtering reliable stations ...")
    pm_matrix = get_pm_matrix(pivot)
    stats_df  = steps.filter_stations(
        pm_matrix,
        min_hours      = config["min_station_hours"],
        min_continuity = config["min_continuity"],
    )
    print(f"   -> {len(stats_df)} stations passed the filter")

    print("4. Filling gaps via spatial neighbors ...")
    pm_filled = steps.fill_gaps(pm_matrix[stats_df["station"]], top_k=config["top_k_neighbors"])

    print("5. Selecting target and neighbor stations ...")
    target, related = steps.select_stations(stats_df, pm_filled, top_k=config["top_k_neighbors"])
    print(f"   -> target: {target}")
    print(f"   -> neighbors: {related}")

    print("6. Feature engineering ...")
    X, y = steps.build_features(
        pivot, pm_filled, target, related,
        forecast_h   = config["forecast_horizon_h"],
        include_pm10 = config["use_pm10_feature"],
    )
    print(f"   -> feature matrix: {X.shape}")

    print("7. Splitting into training / test sets ...")
    X_train, X_test, y_train, y_test = steps.split(X, y, test_size=config["test_size"])
    print(f"   -> training: {len(X_train):,}  |  test: {len(X_test):,}")

    print("8. Training models ...")
    models = steps.train(X_train, y_train, tscv)

    print("9. Evaluation ...")
    metrics = eval_fn(models, X_test, y_test)
    print(metrics)

    print("\nML pipeline completed successfully.")
    return MLPipelineResult(
        models           = models,
        metrics          = metrics,
        X_train          = X_train,
        X_test           = X_test,
        y_train          = y_train,
        y_test           = y_test,
        target_station   = target,
        related_stations = related,
    )

In [ ]:
# -- Execution ----------------------------------------------------------------
# result = run_ml_pipeline(df, CONFIG)
# plot_error_analysis(result.models, result.X_test, result.y_test)
# plot_feature_importance(result.models, result.X_train)
# plot_time_series_comparison(result.models, result.X_test, result.y_test)

## 11. Evaluation and visualization

Next function computes R2, MAE, and RMSE for each model on the test set.

**Parameters**
- models : Dict[str, Any] -- {name: trained estimator}.
- X_test : pd.DataFrame   -- test features.
- y_test : pd.Series      -- true target values.

**Returns**
- pd.DataFrame -- metrics table, sorted by R2 descending.

In [ ]:
def evaluate(models: Dict[str, Any], X_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame:
    rows = []
    for name, model in models.items():
        preds = model.predict(X_test)
        rows.append({
            "Model": name,
            "R²"  : round(r2_score(y_test, preds), 4),
            "MAE" : round(mean_absolute_error(y_test, preds), 3),
            "RMSE": round(np.sqrt(mean_squared_error(y_test, preds)), 3),
        })
    return pd.DataFrame(rows).set_index("Model").sort_values("R²", ascending=False)


results = evaluate(models, X_test, y_test)
print("\n-- Model Performance -------------------------------------------------")
print(results)

### 11.1 Actual vs. forecast and distribution of balances

Side-by-side plot: scatter plot of actual vs predicted and residual histogram.

**Parameters**
- models : Dict[str, Any] -- must contain key 'Ensemble'.
- X_test : pd.DataFrame
- y_test : pd.Series

In [ ]:
def plot_error_analysis(
    models : Dict[str, Any],
    X_test : pd.DataFrame,
    y_test : pd.Series,
) -> None:
    model = models["Ensemble"]
    preds = model.predict(X_test)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # -- Scatter: actual vs predicted -----------------------------------------
    sns.scatterplot(x=y_test, y=preds, alpha=0.4, ax=ax1)
    lims = [y_test.min(), y_test.max()]
    ax1.plot(lims, lims, "r--", label="Perfect prediction")
    ax1.set_title(
        f"Actual vs Predicted PM2.5 (Ensemble)\n"
        f"MAE = {mean_absolute_error(y_test, preds):.2f} µg/m³"
    )
    ax1.set_xlabel("Actual PM2.5")
    ax1.set_ylabel("Predicted PM2.5")
    ax1.legend()
    ax1.grid(alpha=0.3)

    # -- Histogram: residuals ------------------------------------------------
    residuals = y_test - preds
    sns.histplot(residuals, kde=True, color="purple", ax=ax2)
    ax2.axvline(0, color="black", linestyle="--")
    ax2.set_title("Residual distribution")
    ax2.set_xlabel("Error (Actual - Predicted)")
    ax2.set_ylabel("Frequency")

    plt.tight_layout()
    plt.savefig("error_analysis.png", dpi=150)
    plt.show()


plot_error_analysis(models, X_test, y_test)

### 11.2 Feature importance (XGBoost)


In [ ]:
def plot_feature_importance(
    models  : Dict[str, Any],
    X_train : pd.DataFrame,
    top_n   : int = 15,
) -> None:
    xgb_model   = models["XGBoost_Tuned"].regressor_.named_steps["model"]
    importances = xgb_model.feature_importances_

    fi_df = (
        pd.DataFrame({"Feature": X_train.columns, "Importance": importances})
        .sort_values("Importance", ascending=False)
        .head(top_n)
    )

    plt.figure(figsize=(10, 8))
    sns.barplot(data=fi_df, x="Importance", y="Feature", palette="viridis")
    plt.title(f"Top-{top_n} most important features (XGBoost)")
    plt.xlabel("Relative importance score")
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150)
    plt.show()


plot_feature_importance(models, X_train)

### 11.3 Comparing Time Series Forecasts -- Final Week

In [ ]:
def plot_time_series_comparison(
    models     : Dict[str, Any],
    X_test     : pd.DataFrame,
    y_test     : pd.Series,
    last_n_h   : int = 168,
) -> None:
    preds = models["Ensemble"].predict(X_test)

    plt.figure(figsize=(15, 5))
    plt.plot(y_test.index[-last_n_h:], y_test.values[-last_n_h:],
             label="Actual", alpha=0.8)
    plt.plot(y_test.index[-last_n_h:], preds[-last_n_h:],
             label="Ensemble (prediction)", linestyle="--")
    plt.title(f"Final week -- Actual vs Predicted PM2.5 ({CONFIG['forecast_horizon_h']} hours ahead)")
    plt.xlabel("Time")
    plt.ylabel("PM2.5 (µg/m³)")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig("timeseries_performance.png", dpi=150)
    plt.show()


plot_time_series_comparison(models, X_test, y_test, last_n_h=168)